# Task 9 - LLM narrative layer
Turns Task 7's segment profiles and Task 8's promo offers into grounded,
plain-language descriptions a marketer or manager can act on without
reading RFM scores or margin formulas themselves.

The LLM is given ONLY the real numbers from those tables and explicitly
instructed not to invent anything beyond them — this is grounded prompt
engineering, not free-form generation.

Logic lives in `src/narratives.py`. Reuses the Gemini setup from Task 4
(`GOOGLE_API_KEY` via `.env`).

In [ ]:
import os, sys

# Notebooks live in Jupyters/, code lives in src/  ->  always run from the project root
if os.path.basename(os.getcwd()) == "Jupyters":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
assert os.path.exists("olist.db"), f"olist.db not found in {os.getcwd()} - launch Jupyter from the project root"

import pandas as pd
from src.data_loader import get_connection

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
conn = get_connection()

In [ ]:
from src.narratives import *

## 1. Rebuild Task 7's segment profile and Task 8's offers
(same inputs Task 9 narrates — nothing new computed here)

In [ ]:
from src.segmentation import compute_rfm, add_segments, segment_profile
from src.promo_generator import build_offers, Assumptions

profile = segment_profile(add_segments(compute_rfm(conn, merge_gap_hours=1.0)))
offers = build_offers(conn, Assumptions())

display(profile)
display(offers[["segment", "recommendation", "status"]])

## 2. Segment descriptions
One LLM call per segment, grounded strictly in `profile`'s numbers.

In [ ]:
segment_descriptions = generate_all_segment_descriptions(profile)

for seg, desc in segment_descriptions.items():
    print(f"--- {seg} ---")
    print(desc)
    print()

## 3. Promo rationales
One LLM call per segment, grounded strictly in `offers`'s numbers — this is
what makes Task 8's human-approval step fast to actually use: instead of
reading `net_per_customer_low/base/high` and `break_even_uplift` directly,
the approver reads one paragraph that already did that translation.

In [ ]:
promo_rationales = generate_all_promo_rationales(offers)

for seg, rationale in promo_rationales.items():
    print(f"--- {seg} ---")
    print(rationale)
    print()

## 4. Sanity-check for hallucination
Pick one segment and manually verify every claim in its description traces
back to a real number above — this is the actual quality check for grounded
prompting, not just "does it read well".

In [ ]:
check_segment = "Champions"
print("Raw numbers:")
print(profile.loc[check_segment])
print("\nGenerated description:")
print(segment_descriptions[check_segment])

## 5. Save narratives for the write-up / Streamlit app

In [ ]:
os.makedirs("reports", exist_ok=True)

with open("reports/segment_narratives.md", "w") as f:
    for seg, desc in segment_descriptions.items():
        f.write(f"## {seg}\n{desc}\n\n")

with open("reports/promo_narratives.md", "w") as f:
    for seg, rationale in promo_rationales.items():
        f.write(f"## {seg}\n{rationale}\n\n")

print("saved reports/segment_narratives.md and reports/promo_narratives.md")

### Limits to state in the demo
* Narratives are only as accurate as the numbers feeding them — if Task 7/8
  numbers change (e.g. from a different `merge_gap_hours` or margin
  assumption), these must be regenerated, not reused.
* Always spot-check a few outputs (Section 4) — grounded prompting reduces
  hallucination risk but doesn't eliminate it entirely.
* These are single-turn generations, not a conversational agent — each
  narrative is generated independently, same scope discipline as Task 4's
  SQL agent.

Feeds **Task 10** — these narratives are what the Streamlit app displays
alongside the raw tables.